In [1]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer
import torch
import json
from peft import prepare_model_for_kbit_training

g:\AI\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
g:\AI\ai_env\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
MODEL_NAME = "Qwen/Qwen3-8B"


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [5]:
bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16,

    bnb_4bit_use_double_quant=True
)

In [15]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards: 100%|██████████| 5/5 [01:19<00:00, 15.95s/it]


In [14]:
dataset = load_dataset(
    "json",
    data_files="./V5C/V5C.jsonl",
)

train_dataset = dataset["train"]

print("Training samples:", len(train_dataset))

Generating train split: 73950 examples [00:00, 192900.27 examples/s]

Training samples: 73950


V5C DATASET CLEANING
Input : V5C-33K-addOn-Isha.jsonl
Output: G:\V5-dataset\V5C\V5C_clean.jsonl

CLEANING COMPLETE
Total records      : 42134
Valid records      : 41971
Removed records    : 163
JSON errors        : 0
Schema errors      : 163
Valid percentage   : 99.61%
Removed percentage : 0.39%

REMOVED SAMPLES
Line     1140 : Wrong top-level keys: ['instruction', 'input']
Line     1141 : Wrong top-level keys: ['instruction', 'input']
Line     1142 : Wrong top-level keys: ['instruction', 'input']
Line     1143 : Wrong top-level keys: ['instruction', 'input']
Line     1983 : Wrong input keys: ['relationship', 'Conversation']
Line     1985 : Wrong top-level keys: ['instruction', 'input']
Line     1986 : Wrong top-level keys: ['instruction', 'input']
Line     1987 : Wrong top-level keys: ['instruction', 'input']
Line     1988 : Wrong top-level keys: ['instruction', 'input']
Line     1989 : Wrong top-level keys: ['instruction', 'input']
Line     1990 : Wrong top-level keys: ['instruction'

In [11]:
import json

# ==========================================================
# V5C SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """You are an expert conversation understanding assistant.

Your task is to summarize the given conversation accurately.

Rules:
- Summarize only the information present in the conversation.
- Do NOT invent information.
- Do NOT add unsupported details.
- Preserve the main events, decisions, actions, and important context.
- Keep the summary concise and factual.
- Do NOT mention information that is not supported by the conversation.

Return ONLY valid JSON in the following format:

{
  "summary": ""
}

Field Definition:
- summary: A concise summary of the conversation, normally 1–3 sentences.

Return only the JSON object. Do not include markdown or extra text.
"""


# ==========================================================
# V5C FORMATTING FUNCTION
# ==========================================================

def formatting_func(example):

    # ------------------------------------------------------
    # User input
    # ------------------------------------------------------
    user_input = (
        f"{example['instruction']}\n\n"
        f"{json.dumps(example['input'], ensure_ascii=False, separators=(',', ':'))}"
    )

    # ------------------------------------------------------
    # Assistant output
    # ------------------------------------------------------
    assistant_output = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":")
    )

    # ------------------------------------------------------
    # Chat messages
    # ------------------------------------------------------
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_input
        },
        {
            "role": "assistant",
            "content": assistant_output
        }
    ]

    # ------------------------------------------------------
    # Apply tokenizer chat template
    # ------------------------------------------------------
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )


# ==========================================================
# TEST ONE SAMPLE
# ==========================================================

formatted_sample = formatting_func(train_dataset[0])

print("=" * 80)
print("V5C FORMATTING TEST")
print("=" * 80)
print(formatted_sample)
print("=" * 80)

V5C FORMATTING TEST
<|im_start|>system
You are an expert conversation understanding assistant.

Your task is to summarize the given conversation accurately.

Rules:
- Summarize only the information present in the conversation.
- Do NOT invent information.
- Do NOT add unsupported details.
- Preserve the main events, decisions, actions, and important context.
- Keep the summary concise and factual.
- Do NOT mention information that is not supported by the conversation.

Return ONLY valid JSON in the following format:

{
  "summary": ""
}

Field Definition:
- summary: A concise summary of the conversation, normally 1–3 sentences.

Return only the JSON object. Do not include markdown or extra text.
<|im_end|>
<|im_start|>user
Summarize the conversation

{"relationship":"Colleague","conversation":"Colleague: Hey, do you have a minute to talk about the Q3 marketing budget?\nUser: Sure, I just finished looking over the spreadsheet. What's on your mind?\nColleague: We need to trim about fifte

In [15]:
# Printing  TOKEN ,DISTRIBUTION ,FORMATTING ,LONGEST ,SAMPLES of this dataset (important thing before training)

import time
from statistics import mean, median

def analyze_dataset(name, dataset, tokenizer, formatting_func):
    print("\n" + "="*70)
    print(name)
    print("="*70)

    train = dataset["train"]

    lengths = []
    format_times = []

    start_total = time.time()

    for i, sample in enumerate(train):
        t1 = time.time()

        text = formatting_func(sample)

        t2 = time.time()
        format_times.append(t2 - t1)

        tokens = tokenizer(text, add_special_tokens=True)["input_ids"]
        lengths.append(len(tokens))

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(train)}")

    total_time = time.time() - start_total

    print("\n----- BASIC -----")
    print("Samples              :", len(train))
    print("Average Tokens       :", round(mean(lengths),2))
    print("Median Tokens        :", median(lengths))
    print("Minimum Tokens       :", min(lengths))
    print("Maximum Tokens       :", max(lengths))

    print("\n----- TOKEN DISTRIBUTION -----")
    print(">256 tokens          :", sum(x > 256 for x in lengths))
    print(">512 tokens          :", sum(x > 512 for x in lengths))
    print(">1024 tokens         :", sum(x > 1024 for x in lengths))
    print(">2048 tokens         :", sum(x > 2048 for x in lengths))
    print(">4096 tokens         :", sum(x > 4096 for x in lengths))

    print("\n----- FORMATTING -----")
    print("Formatting Time      :", round(total_time,2), "sec")
    print("Average/sample       :", round(mean(format_times)*1000,3), "ms")
    print("Samples/sec          :", round(len(train)/total_time,2))

    print("\n----- LONGEST SAMPLES -----")
    top = sorted(enumerate(lengths), key=lambda x: x[1], reverse=True)[:10]

    for idx, tok in top:
        print(f"Sample {idx:6d} : {tok} tokens")

    return lengths


In [16]:
old_dataset = load_dataset(
    "json",
    data_files= r"G:\V5-dataset\V5C\V5C.jsonl",
  
)

# new_dataset = load_dataset(
#     "json",
#     data_files=r"./newV4a/finalV4A.jsonl",
  
# )


old_lengths = analyze_dataset(
    "OLD DATASET",
    old_dataset,
    tokenizer,
    formatting_func
)

# new_lengths = analyze_dataset(
#     "NEW V4A DATASET",
#     new_dataset,
#     tokenizer,
#     formatting_func
# )

Generating train split: 73950 examples [00:00, 180809.62 examples/s]



OLD DATASET
Processed 5000/73950
Processed 10000/73950
Processed 15000/73950
Processed 20000/73950
Processed 25000/73950
Processed 30000/73950
Processed 35000/73950
Processed 40000/73950
Processed 45000/73950
Processed 50000/73950
Processed 55000/73950
Processed 60000/73950
Processed 65000/73950
Processed 70000/73950

----- BASIC -----
Samples              : 73950
Average Tokens       : 410.55
Median Tokens        : 405.0
Minimum Tokens       : 254
Maximum Tokens       : 888

----- TOKEN DISTRIBUTION -----
>256 tokens          : 73949
>512 tokens          : 4499
>1024 tokens         : 0
>2048 tokens         : 0
>4096 tokens         : 0

----- FORMATTING -----
Formatting Time      : 96.07 sec
Average/sample       : 0.21 ms
Samples/sec          : 769.73

----- LONGEST SAMPLES -----
Sample  17172 : 888 tokens
Sample   6526 : 851 tokens
Sample   3281 : 828 tokens
Sample  55399 : 826 tokens
Sample  66131 : 826 tokens
Sample  33993 : 823 tokens
Sample  24807 : 821 tokens
Sample  67405 : 819

Code for removeing bad example very helpfull


In [ ]:
import json
import os

# ============================================================
# PATHS
# ============================================================

INPUT_PATH = r"V5C-33K-addOn-Isha.jsonl"

OUTPUT_PATH = r"G:\V5-dataset\V5C\V5C_clean.jsonl"

# ============================================================
# COUNTERS
# ============================================================

total = 0
valid = 0
removed = 0

json_errors = 0
schema_errors = 0

bad_samples = []


# ============================================================
# VALIDATE V5C SAMPLE
# ============================================================

def validate_v5c(sample):
    """
    Validate one V5C dataset sample.

    Expected structure:

    {
        "instruction": "Summarize the conversation",

        "input": {
            "relationship": "...",
            "conversation": "..."
        },

        "output": {
            "summary": "..."
        }
    }
    """

    # --------------------------------------------------------
    # Top-level must be dict
    # --------------------------------------------------------

    if not isinstance(sample, dict):
        return False, "Top-level record is not a dictionary"

    # --------------------------------------------------------
    # Exact top-level keys
    # --------------------------------------------------------

    required_top_keys = {
        "instruction",
        "input",
        "output"
    }

    if set(sample.keys()) != required_top_keys:
        return False, (
            f"Wrong top-level keys: {list(sample.keys())}"
        )

    # --------------------------------------------------------
    # instruction
    # --------------------------------------------------------

    if not isinstance(sample["instruction"], str):
        return False, "instruction is not a string"

    if not sample["instruction"].strip():
        return False, "instruction is blank"

    # --------------------------------------------------------
    # input
    # --------------------------------------------------------

    if not isinstance(sample["input"], dict):
        return False, "input is not a dictionary"

    required_input_keys = {
        "relationship",
        "conversation"
    }

    if set(sample["input"].keys()) != required_input_keys:
        return False, (
            f"Wrong input keys: {list(sample['input'].keys())}"
        )

    # relationship
    if not isinstance(
        sample["input"]["relationship"],
        str
    ):
        return False, "input.relationship is not a string"

    if not sample["input"]["relationship"].strip():
        return False, "input.relationship is blank"

    # conversation
    if not isinstance(
        sample["input"]["conversation"],
        str
    ):
        return False, "input.conversation is not a string"

    if not sample["input"]["conversation"].strip():
        return False, "input.conversation is blank"

    # --------------------------------------------------------
    # output
    # --------------------------------------------------------

    if not isinstance(sample["output"], dict):
        return False, "output is not a dictionary"

    required_output_keys = {
        "summary"
    }

    if set(sample["output"].keys()) != required_output_keys:
        return False, (
            f"Wrong output keys: {list(sample['output'].keys())}"
        )

    # summary
    if not isinstance(
        sample["output"]["summary"],
        str
    ):
        return False, "output.summary is not a string"

    if not sample["output"]["summary"].strip():
        return False, "output.summary is blank"

    # --------------------------------------------------------
    # Everything is correct
    # --------------------------------------------------------

    return True, None


# ============================================================
# PROCESS DATASET
# ============================================================

print("=" * 70)
print("V5C DATASET CLEANING")
print("=" * 70)

print(f"Input : {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print()


with open(
    INPUT_PATH,
    "r",
    encoding="utf-8"
) as infile, open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as outfile:

    for line_number, line in enumerate(
        infile,
        start=1
    ):

        # ----------------------------------------------------
        # Skip blank lines
        # ----------------------------------------------------

        if not line.strip():
            continue

        total += 1

        # ----------------------------------------------------
        # JSON validation
        # ----------------------------------------------------

        try:
            sample = json.loads(line)

        except json.JSONDecodeError as e:

            json_errors += 1
            removed += 1

            bad_samples.append({
                "line": line_number,
                "reason": f"Invalid JSON: {e}"
            })

            continue

        # ----------------------------------------------------
        # Schema validation
        # ----------------------------------------------------

        is_valid, reason = validate_v5c(sample)

        if not is_valid:

            schema_errors += 1
            removed += 1

            bad_samples.append({
                "line": line_number,
                "reason": reason
            })

            continue

        # ----------------------------------------------------
        # Valid sample
        # ----------------------------------------------------

        outfile.write(
            json.dumps(
                sample,
                ensure_ascii=False,
                separators=(",", ":")
            ) + "\n"
        )

        valid += 1


# ============================================================
# PRINT RESULTS
# ============================================================

print("=" * 70)
print("CLEANING COMPLETE")
print("=" * 70)

print(f"Total records      : {total}")
print(f"Valid records      : {valid}")
print(f"Removed records    : {removed}")
print(f"JSON errors        : {json_errors}")
print(f"Schema errors      : {schema_errors}")

print("=" * 70)

if total > 0:
    print(
        f"Valid percentage   : "
        f"{valid / total * 100:.2f}%"
    )

    print(
        f"Removed percentage : "
        f"{removed / total * 100:.2f}%"
    )

print("=" * 70)


# ============================================================
# SHOW BAD SAMPLES
# ============================================================

if bad_samples:

    print()
    print("=" * 70)
    print("REMOVED SAMPLES")
    print("=" * 70)

    for item in bad_samples[:50]:

        print(
            f"Line {item['line']:>8} : "
            f"{item['reason']}"
        )

    if len(bad_samples) > 50:

        print()
        print(
            f"... and "
            f"{len(bad_samples) - 50} more removed samples"
        )

else:

    print()
    print("✓ No problematic samples found")


# ============================================================
# CHECK OUTPUT FILE EXISTS
# ============================================================

print()
print("=" * 70)

if os.path.exists(OUTPUT_PATH):

    size_mb = (
        os.path.getsize(OUTPUT_PATH)
        / (1024 * 1024)
    )

    print("✓ Clean dataset created successfully")
    print(f"File : {OUTPUT_PATH}")
    print(f"Size : {size_mb:.2f} MB")

else:

    print("✗ Output file was not created")

print("=" * 70)